# Initialize and import packages

In [ ]:
# Import required libraries for data processing, geospatial analysis, and Earth Engine operations
import geopandas as gpd  # Geospatial data manipulation
import pandas as pd  # Data manipulation and analysis
import math  # Mathematical operations
import ee  # Google Earth Engine Python API
import geemap  # Earth Engine mapping tools
import os  # Operating system interface
import json  # JSON data handling
import ast  # Abstract Syntax Tree parsing

In [ ]:
# Authenticate with Google Earth Engine and initialize the project
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='ee-curuai2')

In [ ]:
# Optional: Mount Google Drive for Colab environment
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Set the working directory where water period definitions are stored
wrk_directory = r"C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Water Period Definitions"

# Import Landsat Image Collections

## Landsat 5 and Landsat 7 - Atmospheric Correction with PY6S

In [ ]:
# Load Landsat 5 and 7 atmospheric corrected data from multiple Earth Engine assets
# All Landsat 5/7 data has been processed with PY6S atmospheric correction
landsat5 = ee.ImageCollection("projects/ee-curuai2/assets/Py6S/LD5/Landsat5")\
            .merge(ee.ImageCollection("projects/ee-curuai/assets/Py6S/LD7/Landsat7"))\
            .merge(ee.ImageCollection('projects/ee-curuai2/assets/Py6S/LD7/Landsat7'))\
            .select(['B1', 'B2', 'B3', 'B4', 'B5', 'B7'])  # Select relevant spectral bands

# Display temporal range and total number of images
print(ee.Date(landsat5.sort('system:time_start', True).first().get('system:time_start')).format().getInfo())
print(ee.Date(landsat5.sort('system:time_start', False).first().get('system:time_start')).format().getInfo())
print(landsat5.size().getInfo())

1985-02-07T13:24:25
2024-01-16T11:16:49
2446


## Landsat 8 and Landsat 9 - Atmospheric Correction with PY6S

In [ ]:
# Load Landsat 8 and 9 atmospheric corrected data from multiple Earth Engine assets
# All Landsat 8/9 data has been processed with PY6S atmospheric correction
landsat8 = (ee.ImageCollection("projects/ee-curuai/assets/Py6S/LD8/Landsat8")
            .merge(ee.ImageCollection('projects/ee-curuai2/assets/Py6S/LD9/Landsat9'))
            .select(['B2', 'B3', 'B4', 'B5', 'B6', 'B7']))  # Select relevant spectral bands

# Display temporal range and total number of images
print(ee.Date(landsat8.sort('system:time_start', True).first().get('system:time_start')).format().getInfo())
print(ee.Date(landsat8.sort('system:time_start', False).first().get('system:time_start')).format().getInfo())
print(landsat8.size().getInfo())

2013-05-11T13:55:54
2025-12-15T13:48:04
1081


# Convert to Remote Sensing Reflectance and Apply Sunglint Correction

In [ ]:
def deglint(img):
    """
    Apply deglint correction to convert radiance to remote sensing reflectance (Rrs).
    
    Process:
    1. Convert radiance to reflectance by dividing by pi: Rrs_sat_ac = Rsat_ac / π
    2. Apply deglint correction by subtracting SWIR band from visible/NIR bands
       Rrs_sat_ac_deglint(VNIR) = Rrs_sat_ac(VNIR) - Rrs_sat_ac(SWIR)
    3. Correction method follows INPE CURUAI publication methodology
    
    Parameters:
    - img: Input image with atmospheric corrected data
    
    Returns:
    - Corrected image with deglinted reflectance values and preserved metadata
    """
    # Divide by pi to convert from radiance to reflectance
    Rrs = img.divide(math.pi)
    
    # Subtract SWIR from all bands to remove sunglint effects
    deglint = Rrs.select(['blue_mean', 'green_mean', 'red_mean', 'nir_mean', 'swir1', 'swir2']) \
        .subtract(Rrs.select('swir1'))

    # Copy relevant metadata from original image
    return (deglint
            .copyProperties(img, ['system:time_start', 'CLOUD_COVER', "system:index"]))

## Standardize Band Names Across Landsat Sensors

In [ ]:
# Define standardized band names to match across different Landsat sensors
# This allows consistent processing of Landsat 5/7/8/9 data
name_bands = ['blue_mean', 'green_mean', 'red_mean', 'nir_mean', 'swir1', 'swir2']

### Landsat 5 and 7 Band Renaming

In [ ]:
# Rename Landsat 5/7 bands to standardized names for consistent processing
ld5 = landsat5.map(lambda img: img.rename(name_bands))

# Display total number of images in collection
display(ld5.size().getInfo())

2446

### Landsat 8 and 9 Band Renaming

In [ ]:
# Rename Landsat 8/9 bands to standardized names for consistent processing
ld8 = landsat8.map(lambda img: img.rename(name_bands))

# Display total number of images in collection
display(ld8.size().getInfo())

1081

In [ ]:
# Merge Landsat 5/7 and 8/9 collections, sort chronologically, and apply deglint correction
merge_col = ld5.merge(ld8).sort('system:time_start').map(deglint)

# Display first 5 images to verify processing
merge_col.limit(5)

In [ ]:
# Display merged collection temporal range and total image count
print(ee.Date(merge_col.sort('system:time_start', True).first().get('system:time_start')).format().getInfo())
print(ee.Date(merge_col.sort('system:time_start', False).first().get('system:time_start')).format().getInfo())
print(merge_col.size().getInfo())

1985-02-07T13:24:25
2025-12-15T13:48:04
3527


In [ ]:
# Create interactive map to visualize median composite of all merged images
Map = geemap.Map()
median = merge_col.median()
Map.addLayer(median, {"bands": ['red_mean', "green_mean", 'blue_mean'], 'min': 0, 'max': 0.0165}, "Visualization Test")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

# Import Water Period Definitions Based on Óbidos Water Level

In [ ]:
# Load water period definitions generated from hydrological analysis
df_period_limits = pd.read_csv(os.path.join(wrk_directory, 'water_period_limits.csv')).drop(columns=['Unnamed: 0'])

# Convert date columns to datetime format for proper temporal processing
df_period_limits['lim_next'] = pd.to_datetime(df_period_limits['lim_next'])
df_period_limits['lim_previous'] = pd.to_datetime(df_period_limits['lim_previous'])

# Sort by start date to ensure chronological order
df_period_limits = df_period_limits.sort_values(by='lim_next').reset_index(drop=True)

# Fill missing end dates with 3-month offset from start date
df_period_limits['lim_next'] = df_period_limits['lim_next'].fillna(
    df_period_limits['lim_previous'] + pd.DateOffset(months=3)
)

# Fill missing start dates with negative 3-month offset from end date
df_period_limits['lim_previous'] = df_period_limits['lim_previous'].fillna(
    df_period_limits['lim_next'] + pd.DateOffset(months=-3)
)

# Display last rows to verify data completeness
display(df_period_limits.tail())

,lim_previous,lim_next,water_period
220,2024-03-16 12:00:00,2024-06-08 06:00:00,HW
221,2024-06-08 06:00:00,2024-08-23 18:00:00,F
222,2024-08-23 18:00:00,2024-11-23 00:00:00,LW
223,2024-11-23 00:00:00,2025-03-09 00:00:00,R
224,2025-03-09 00:00:00,2025-06-09 00:00:00,HW


In [ ]:
# Define additional periods for recent data not covered by historical classification
fill_periods = {
    'lim_previous': ['2025-06-09 00:00:00', '2025-09-09 00:00:00', '2025-12-09 00:00:00'],
    "lim_next": ['2025-09-09 00:00:00', '2025-12-09 00:00:00', '2026-01-01 00:00:00'],
    'water_period': ['F', 'LW', "R"]  # F=Falling, LW=Low Water, R=Rising
}
fill_periods = pd.DataFrame(fill_periods)

# Append new periods to the dataframe and reset index
df_period_limits = pd.concat([df_period_limits, fill_periods]).reset_index(drop=True)

# Ensure date columns are properly formatted as datetime
df_period_limits['lim_previous'] = pd.to_datetime(df_period_limits['lim_previous'])
df_period_limits['lim_next'] = pd.to_datetime(df_period_limits['lim_next'])

# Extract year and month from start date for later reference
df_period_limits['year'] = df_period_limits['lim_previous'].dt.year
df_period_limits['month'] = df_period_limits['lim_previous'].dt.month

# Display the complete period limits table
df_period_limits

,lim_previous,lim_next,water_period,year,month
0,1968-06-30 12:00:00,1968-09-30 12:00:00,HW,1968,6
1,1968-09-30 12:00:00,1969-08-01 12:00:00,F,1968,9
2,1969-08-01 12:00:00,1970-02-07 18:00:00,LW,1969,8
3,1970-02-07 18:00:00,1970-04-24 06:00:00,R,1970,2
4,1970-04-24 06:00:00,1970-07-16 18:00:00,HW,1970,4
...,...,...,...,...,...
223,2024-11-23 00:00:00,2025-03-09 00:00:00,R,2024,11
224,2025-03-09 00:00:00,2025-06-09 00:00:00,HW,2025,3
225,2025-06-09 00:00:00,2025-09-09 00:00:00,F,2025,6
226,2025-09-09 00:00:00,2025-12-09 00:00:00,LW,2025,9


# Create Image Mosaics Grouped by Water Period (Based on Óbidos Water Level)

In [ ]:
# Create median composite mosaics for each water period
list_images = []

# Iterate through each water period definition
for i in range(0, len(df_period_limits)):
    
    # Convert dates to strings formatted for Earth Engine
    start_date = df_period_limits['lim_previous'][i].strftime('%Y-%m-%d')
    end_date = df_period_limits['lim_next'][i].strftime('%Y-%m-%d')

    # Filter images that fall within the period date range
    filter_dates = merge_col.filterDate(ee.Date(start_date), ee.Date(end_date))

    # Check if any images exist in this date range
    if filter_dates.size().getInfo() > 0:
        # Create median composite from all images in the period
        image = filter_dates.median()

        # Verify image has bands before adding to list
        if image.bandNames().size().getInfo() > 0:
            list_images.append(image
                .set({
                    'year': str(df_period_limits['year'][i]),
                    'month_init': str(df_period_limits['month'][i]),
                    'system:time_start': ee.Date(start_date).millis(),  # Milliseconds for system timestamp
                    'system:time_end': ee.Date(end_date).millis(),
                    'time_start': start_date,
                    'time_finish': end_date,
                    'band_count': image.bandNames().length(),
                    "water_period": str(df_period_limits['water_period'][i])  # Classification: R, F, HW, LW
                })
            )
            print("Mosaic created: ", i)
    else:
        # Skip periods with no available imagery
        continue

Mosaic:  63
Mosaic:  64
Mosaic:  65
Mosaic:  66
Mosaic:  68
Mosaic:  69
Mosaic:  70
Mosaic:  71
Mosaic:  72
Mosaic:  73
Mosaic:  74
Mosaic:  75
Mosaic:  76
Mosaic:  77
Mosaic:  78
Mosaic:  79
Mosaic:  80
Mosaic:  81
Mosaic:  82
Mosaic:  83
Mosaic:  84
Mosaic:  85
Mosaic:  86
Mosaic:  87
Mosaic:  88
Mosaic:  89
Mosaic:  90
Mosaic:  91
Mosaic:  92
Mosaic:  93
Mosaic:  94
Mosaic:  95
Mosaic:  96
Mosaic:  97
Mosaic:  98
Mosaic:  99
Mosaic:  100
Mosaic:  101
Mosaic:  102
Mosaic:  103
Mosaic:  104
Mosaic:  105
Mosaic:  106
Mosaic:  107
Mosaic:  108
Mosaic:  109
Mosaic:  110
Mosaic:  111
Mosaic:  112
Mosaic:  113
Mosaic:  114
Mosaic:  116
Mosaic:  117
Mosaic:  118
Mosaic:  119
Mosaic:  120
Mosaic:  121
Mosaic:  122
Mosaic:  123
Mosaic:  124
Mosaic:  125
Mosaic:  126
Mosaic:  127
Mosaic:  128
Mosaic:  129
Mosaic:  130
Mosaic:  131
Mosaic:  132
Mosaic:  133
Mosaic:  134
Mosaic:  135
Mosaic:  136
Mosaic:  137
Mosaic:  138
Mosaic:  139
Mosaic:  140
Mosaic:  141
Mosaic:  142
Mosaic:  143
Mosaic:  

In [ ]:
# Create interactive map to visualize a specific water period mosaic (index 160)
Map = geemap.Map()
median = list_images[160]
Map.add_basemap("Hybrid")
Map.addLayer(median, {"bands": ['red_mean', "green_mean", 'blue_mean'], 'min': 0, 'max': 0.0175}, "Visualization Test")

# Center map on Curuai floodplain area
Map.centerObject(ee.Geometry.Polygon(
        [[[-55.89833867930371, -1.9982415316174194],
          [-55.89833867930371, -2.3646438808303256],
          [-55.08123052500684, -2.3646438808303256],
          [-55.08123052500684, -1.9982415316174194]]], None, False), 10)
Map

In [ ]:
# Remove cloudy and low-quality images from the mosaic list
# Indices correspond to images with excessive cloud cover or artifacts
indices_to_remove = {12, 16, 24, 40, 42, 51, 53, 110, 114, 162, 4, 34, 35, 47, 106}
updated_list = [image for index, image in enumerate(list_images) if index not in indices_to_remove]

# Convert Python list to Earth Engine ImageCollection
period_mosaic = ee.ImageCollection(updated_list)

# Display summary of created mosaic collection
print(f"Created mosaic collection with {period_mosaic.size().getInfo()} images")

Created mosaic with 148 images


# Calculate Water Surface Area for Each Mosaic

In [ ]:
# Function to mask land and extract water pixels using HSV color space
def hsvComposite(image):
    """
    Create water mask using Hue-Saturation-Value (HSV) color space.
    Water pixels typically have hue values between 0.3 and 0.9.
    
    Parameters:
    - image: Input image with RGB bands
    
    Returns:
    - Masked image with only water pixels and selected bands
    """
    # Convert RGB to HSV color space for better water discrimination
    composite = image.select(['blue_mean', 'green_mean', 'red_mean']).rgbToHsv()
    
    # Extract hue band for analysis
    hue = composite.select("hue")
    
    # Create masks based on hue thresholds (water-specific ranges)
    max_mask = hue.lte(0.9)  # Upper hue threshold
    min_mask = hue.gte(0.3)  # Lower hue threshold
    
    # Apply masks and select relevant bands
    return image.updateMask(max_mask).updateMask(min_mask).select(['blue_mean', 'green_mean', 'red_mean', 'nir_mean'])

In [ ]:
def area_calc(img):
    """
    Calculate water surface area within floodplain boundaries.
    
    Process:
    1. Get pixel area in square meters
    2. Apply water mask using HSV color space
    3. Multiply water mask by pixel area
    4. Sum all water pixel areas and convert to km²
    
    Parameters:
    - img: Input image with spectral bands
    
    Returns:
    - Image with 'area_km2' property added
    """
    # Get pixel area in square meters (30m x 30m for Landsat)
    pixel_area = ee.Image.pixelArea()

    # Load floodplain boundary
    floodplain = ee.FeatureCollection('projects/ee-curuai2/assets/varzea_alagavel')
    
    # Apply water mask to image
    image = hsvComposite(img)
    
    # Create binary water mask (1 for water, 0 for non-water)
    img_mask = image.gt(0)

    # Calculate area by multiplying mask by pixel area
    areaImage = img_mask.multiply(pixel_area)

    # Sum all water pixel areas within floodplain
    area = areaImage.reduceRegion(**{
        'reducer': ee.Reducer.sum(),
        'geometry': floodplain.geometry(),
        'scale': 30,
        'maxPixels': 1e10
    })
    
    # Convert from m² to km² (divide by 1 million)
    return image.set('area_km2', ee.Number(area.get('red_mean')).divide(1e6))

In [ ]:
# Calculate water surface area for each period mosaic
period_area = period_mosaic.map(area_calc)

# Display first 3 images to verify processing
period_area.limit(3)

In [ ]:
# Visualize the first water period mosaic with area calculation
Map = geemap.Map()
median = period_area.first()
Map.add_basemap("Hybrid")
Map.addLayer(median, {"bands": ['red_mean', "green_mean", 'blue_mean'], 'min': 0, 'max': 0.0175}, "Visualization Test")

# Center map on Curuai floodplain area
Map.centerObject(ee.Geometry.Polygon(
        [[[-55.89833867930371, -1.9982415316174194],
          [-55.89833867930371, -2.3646438808303256],
          [-55.08123052500684, -2.3646438808303256],
          [-55.08123052500684, -1.9982415316174194]]], None, False), 10)
Map

# Apply Machine Learning Model and Classify TSS (Total Suspended Sediment)

In [ ]:
# Set directory paths for data sources
data_directory = r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Landsat Sampling\Merged Landsat Data'
model_directory = "C:/Users/l_v_v/Documents/GitHub/time_series_curuai/datasets/Parameters Time series/TSS Modeling"

In [ ]:
# Load the best performing model configuration from previous analysis
model_info = pd.read_csv(os.path.join(model_directory, 'model_selection.csv')).iloc[0]

# Extract model parameters and set random seed for reproducibility
params = ast.literal_eval(model_info['Params'])
params['seed'] = 123

# Display model parameters
params

{'numberOfTrees': 500,
 'shrinkage': None,
 'samplingRate': 0.6,
 'loss': 'Huber',
 'seed': 123}

In [ ]:
# Load training data with field measurements and satellite reflectance
df_data = pd.read_csv(os.path.join(data_directory, 'min_date.csv'))

# Remove unnecessary columns and rename SPM to TSS for consistency
df_data = df_data.drop(columns=['Unnamed: 0', 'CHLOROPHYLL',
                      'CHLOROPHYLL_B', 'DOC', 'dif_date_point',
                      'N_TOTAL', 'N_TOTAL_DISSOLVED',
                      'POC', 'P_ORGANIC', 'P_TOTAL',
                      'SILICA', 'TOC', 'duplicated'], axis=1).rename(columns={'SPM': "TSS"})

# Select only the features needed for the TSS prediction model
df_subset = df_data[['TSS', 'blue_mean',
       'green_mean',
       'nir_mean',
       'red_mean',
       'datetime',
       'WATER_PERIOD',
       'LOCATION',
       'LONGITUDE',
       'LATITUDE']].copy()

# Remove rows with missing values
df_subset = df_subset.dropna()
print(df_subset.isna().sum())  # Display count of missing values

# Extract date from datetime column
df_subset['date'] = df_subset['datetime'].apply(lambda row: row[:10])

# Convert to GeoDataFrame with point geometry for each sampling location
gdf = gpd.GeoDataFrame(
    df_subset, geometry=gpd.points_from_xy(df_subset.LONGITUDE, df_subset.LATITUDE),
    crs="EPSG:4326"
)

# Convert GeoDataFrame to JSON format for Earth Engine processing
dataset_json = gdf.to_json()

# Extract features from JSON and convert to Earth Engine FeatureCollection
reg_data = json.loads(dataset_json)
reg_data = reg_data['features']
reg_data = ee.FeatureCollection(reg_data)

# Display training data size
print(reg_data.size().getInfo())

202


In [ ]:
def estimate_TSS(image):
    """
    Estimate TSS concentration using a trained Gradient Boosting Regression model.
    
    Process:
    1. Define spectral bands to use as predictors
    2. Train Gradient Boosting Regressor on field sample data
    3. Apply trained model to predict TSS for all pixels
    
    Parameters:
    - image: Input Landsat image with spectral bands
    
    Returns:
    - Image with TSS band containing predicted concentration values
    """
    # Define spectral bands to use as model features
    predictors = ['blue_mean', 'green_mean', 'red_mean', 'nir_mean']
    
    # Train classifier with Gradient Boosting Regression algorithm
    trained = (ee.Classifier.smileGradientTreeBoost(**params)
           .train(features=reg_data,
                  classProperty="TSS",  # Target property from field measurements
                  inputProperties=predictors)  # Model input features
           .setOutputMode('REGRESSION'))  # Set to regression mode for continuous values
    
    # Apply trained model to predict TSS for each pixel
    return image.select(predictors).classify(trained).rename("TSS").copyProperties(image)

In [ ]:
# Apply TSS estimation model to all water period mosaics
spm_period_classified = period_area.map(estimate_TSS)

# Display first 5 classified images
spm_period_classified.limit(5)

In [ ]:
# Create interactive map to visualize TSS classification results
Map = geemap.Map()
median = spm_period_classified.first()
Map.add_basemap("Hybrid")
Map.addLayer(median, {"bands": ['TSS'], 'min': 0, 'max': 200}, "TSS Visualization Test")

# Center map on Curuai floodplain area
Map.centerObject(ee.Geometry.Polygon(
        [[[-55.89833867930371, -1.9982415316174194],
          [-55.89833867930371, -2.3646438808303256],
          [-55.08123052500684, -2.3646438808303256],
          [-55.08123052500684, -1.9982415316174194]]], None, False), 10)
Map

# Export Processed Data as Earth Engine Assets

In [ ]:
def export_image_tss(img):
    """
    Export TSS classification image to Earth Engine asset.
    
    Defines geographic region covering the study area and sampling extent,
    applies proper projection reference, and starts the export task.
    
    Parameters:
    - img: Earth Engine Image with TSS predictions
    
    Returns:
    - Original image (function is used for side effect of starting export)
    """
    # Define region as union of study area and sample data extent with buffer
    region = ee.FeatureCollection(ee.List([
        ee.Feature(ee.FeatureCollection('projects/ee-curuai2/assets/bacia_local').geometry().buffer(30)),
        ee.Feature(reg_data.geometry().bounds().buffer(30))
    ])).geometry().bounds()
    
    # Get projection reference from a known Landsat image
    projectionRef = ee.Image("LANDSAT/LC08/C02/T1_TOA/LC08_228062_20130425").select('B4').projection().getInfo()
    
    # Extract unique filename from image index
    fname = ee.String(img.get('system:index')).getInfo()
    
    # Configure and start export task
    export = ee.batch.Export.image.toAsset(
        image=ee.Image(img),
        description='mosaic_' + fname,
        assetId='projects/ee-curuai2/assets/landsat_water_period/tss_mosaic/tss_' + fname,
        region=region.buffer(30).bounds(),
        crs=projectionRef['crs'],
        scale=30,  # 30-meter Landsat pixel size
        maxPixels=1e13
    )

    # Start the export task
    export.start()
    print('Exporting ' + fname + ' --> Done')
    return img

In [ ]:
# Get total number of classified images to export
col_length = spm_period_classified.size().getInfo()

# Note: Cannot use map() function for exports because server-side operations cannot be combined with client-side
# Therefore use loop to iterate and export each image individually
# For large time series, it's recommended to break into parts to avoid Earth Engine export limits
# and prevent issues with simultaneous large-scale exports

for i in range(0, col_length):
    # Convert ImageCollection to list and retrieve image at position i
    list_items = spm_period_classified.toList(col_length)
    img = ee.Image(list_items.get(i))
    
    # Export the TSS classified image
    export_image_tss(img)

exporting 0--->done
exporting 1--->done
exporting 2--->done
exporting 3--->done
exporting 4--->done
exporting 5--->done
exporting 6--->done
exporting 7--->done
exporting 8--->done
exporting 9--->done
exporting 10--->done
exporting 11--->done
exporting 12--->done
exporting 13--->done
exporting 14--->done
exporting 15--->done
exporting 16--->done
exporting 17--->done
exporting 18--->done
exporting 19--->done
exporting 20--->done
exporting 21--->done
exporting 22--->done
exporting 23--->done
exporting 24--->done
exporting 25--->done
exporting 26--->done
exporting 27--->done
exporting 28--->done
exporting 29--->done
exporting 30--->done
exporting 31--->done
exporting 32--->done
exporting 33--->done
exporting 34--->done
exporting 35--->done
exporting 36--->done
exporting 37--->done
exporting 38--->done
exporting 39--->done
exporting 40--->done
exporting 41--->done
exporting 42--->done
exporting 43--->done
exporting 44--->done
exporting 45--->done
exporting 46--->done
exporting 47--->done
ex

In [ ]:
def export_image_area(img):
    """
    Export water period mosaic (with area calculation) to Earth Engine asset.
    
    Defines geographic region covering the study area and sampling extent,
    applies proper projection reference, and starts the export task.
    
    Parameters:
    - img: Earth Engine Image with area property
    
    Returns:
    - Original image (function is used for side effect of starting export)
    """
    # Define region as union of study area and sample data extent with buffer
    region = ee.FeatureCollection(ee.List([
        ee.Feature(ee.FeatureCollection('projects/ee-curuai2/assets/bacia_local').geometry().buffer(30)),
        ee.Feature(reg_data.geometry().bounds().buffer(30))
    ])).geometry().bounds()
    
    # Get projection reference from a known Landsat image
    projectionRef = ee.Image("LANDSAT/LC08/C02/T1_TOA/LC08_228062_20130425").select('B4').projection().getInfo()
    
    # Extract unique filename from image index
    fname = ee.String(img.get('system:index')).getInfo()
    
    # Configure and start export task
    export = ee.batch.Export.image.toAsset(
        image=ee.Image(img),
        description='mosaic_' + fname,
        assetId='projects/ee-curuai2/assets/landsat_water_period/water_period_mosaics/mosaic_' + fname,
        region=region.buffer(30).bounds(),
        crs=projectionRef['crs'],
        scale=30,  # 30-meter Landsat pixel size
        maxPixels=1e13
    )

    # Start the export task
    export.start()
    print('Exporting ' + fname + ' --> Done')
    return img

In [ ]:
# Get total number of water period mosaics to export
col_length = period_area.size().getInfo()

# Note: Cannot use map() function for exports because server-side operations cannot be combined with client-side
# Therefore use loop to iterate and export each image individually
# For large time series, it's recommended to break into parts to avoid Earth Engine export limits
# and prevent issues with simultaneous large-scale exports

for i in range(0, col_length):
    # Convert ImageCollection to list and retrieve image at position i
    list_items = period_area.toList(col_length)
    img = ee.Image(list_items.get(i))
    
    # Export the water period mosaic with area calculation
    export_image_area(img)

exporting 0--->done
exporting 1--->done
exporting 2--->done
exporting 3--->done
exporting 4--->done
exporting 5--->done
exporting 6--->done
exporting 7--->done
exporting 8--->done
exporting 9--->done
exporting 10--->done
exporting 11--->done
exporting 12--->done
exporting 13--->done
exporting 14--->done
exporting 15--->done
exporting 16--->done
exporting 17--->done
exporting 18--->done
exporting 19--->done
exporting 20--->done
exporting 21--->done
exporting 22--->done
exporting 23--->done
exporting 24--->done
exporting 25--->done
exporting 26--->done
exporting 27--->done
exporting 28--->done
exporting 29--->done
exporting 30--->done
exporting 31--->done
exporting 32--->done
exporting 33--->done
exporting 34--->done
exporting 35--->done
exporting 36--->done
exporting 37--->done
exporting 38--->done
exporting 39--->done
exporting 40--->done
exporting 41--->done
exporting 42--->done
exporting 43--->done
exporting 44--->done
exporting 45--->done
exporting 46--->done
exporting 47--->done
ex